# banking77-modernbert

Fine-tune `answerdotai/ModernBERT-base` into a 77-way Banking77 intent classifier.
Run top-to-bottom on **Colab Pro** (GPU runtime). Hugging Face auth is handled by the first code cell (paste your token when prompted). See `MODEL_CARD.md` and `docs/superpowers/` for the design and plan.

## Setup & data

In [ ]:
# CELL 1 - setup (safe to re-run, e.g. after a runtime restart)
%cd /content
import os
if not os.path.isdir("banking77-modernbert"):
    !git clone https://github.com/sukhrobnurali/banking77-modernbert.git
%cd banking77-modernbert
!git pull -q

# Install ONLY the pinned HF libs. requirements.txt floors torch/scikit-learn so this
# does NOT upgrade Colab's preinstalled, CUDA-matched torch. Upgrading torch breaks
# Colab's bundled torchvision ("operator torchvision::nms does not exist").
!pip install -q -r requirements.txt

import numpy as np, torch
from datasets import load_dataset
from transformers import set_seed
import config

set_seed(config.SEED)
print("torch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("bf16 supported:", torch.cuda.is_available() and torch.cuda.is_bf16_supported())

In [ ]:
# CELL 1b - Hugging Face auth (needed for the Hub push in Cell 13)
# Paste your HF token in the widget when prompted (use a 'write' token).
# Nothing is hard-coded or read from a variable, so no token lands in the committed notebook.
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# CELL 2 - load data + label maps
# PolyAI/banking77 is script-based; datasets 4.x reads its parquet files directly.
ds = load_dataset(config.DATASET_ID)
print(ds)

label_feature = ds["train"].features[config.LABEL_COLUMN]
id2label = {i: name for i, name in enumerate(label_feature.names)}
label2id = {name: i for i, name in id2label.items()}
assert len(id2label) == config.NUM_LABELS, f"expected {config.NUM_LABELS} labels, got {len(id2label)}"
print("labels:", config.NUM_LABELS, "| example:", id2label[0])
print("train/test sizes:", len(ds["train"]), len(ds["test"]))

If `load_dataset(config.DATASET_ID)` errors on the loading script under datasets 4.x, load the repo's own parquet files directly (keeps provenance with PolyAI):

```python
ds = load_dataset("parquet", data_files={
    "train": "hf://datasets/PolyAI/banking77/data/train-*.parquet",
    "test":  "hf://datasets/PolyAI/banking77/data/test-*.parquet",
})
```
Lock whichever call returns the 77-label `ClassLabel` feature.

In [ ]:
# CELL 3 - EDA: class counts + token length distribution
from collections import Counter
from transformers import AutoTokenizer

counts = Counter(ds["train"][config.LABEL_COLUMN])
print("train per-class min/max:", min(counts.values()), max(counts.values()))

tok_probe = AutoTokenizer.from_pretrained(config.MODEL_ID)
lengths = [len(x) for x in tok_probe(ds["train"][config.TEXT_COLUMN])["input_ids"]]
p99 = int(np.percentile(lengths, 99))
print(f"token length: mean={np.mean(lengths):.1f} p99={p99} max={max(lengths)}")
assert p99 <= config.MAX_LENGTH, f"raise MAX_LENGTH: p99={p99} > {config.MAX_LENGTH}"

## Split & tokenize

In [ ]:
# CELL 4 - stratified train/val split (test stays untouched)
split = ds["train"].train_test_split(
    test_size=config.VAL_FRACTION,
    stratify_by_column=config.LABEL_COLUMN,
    seed=config.SEED,
)
train_ds, val_ds, test_ds = split["train"], split["test"], ds["test"]
print("train/val/test:", len(train_ds), len(val_ds), len(test_ds))

assert set(val_ds[config.LABEL_COLUMN]) == set(range(config.NUM_LABELS))

if config.SMOKE:
    train_ds = train_ds.select(range(config.SMOKE_TRAIN_SIZE))
    val_ds = val_ds.select(range(min(len(val_ds), config.SMOKE_TRAIN_SIZE)))

In [ ]:
# CELL 5 - tokenize + collator
from transformers import DataCollatorWithPadding
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_ID)

def tokenize(batch):
    return tokenizer(batch[config.TEXT_COLUMN], truncation=True, max_length=config.MAX_LENGTH)

tok_train = train_ds.map(tokenize, batched=True, remove_columns=[config.TEXT_COLUMN])
tok_val   = val_ds.map(tokenize,   batched=True, remove_columns=[config.TEXT_COLUMN])
tok_test  = test_ds.map(tokenize,  batched=True, remove_columns=[config.TEXT_COLUMN])
collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(tok_train)

## Train

Tip: set `SMOKE = True` in `config.py` and run Cells 1-8 once for a seconds-long end-to-end sanity check (confirms the v5 `eval_strategy`/`processing_class` kwargs and the pipeline), then set it back to `False` for the full run.

In [ ]:
# CELL 6 - compute_metrics
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

In [ ]:
# CELL 7 - model
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    config.MODEL_ID,
    num_labels=config.NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    attn_implementation="sdpa",   # safe on T4; switch to "flash_attention_2" on A100/L4
)

In [ ]:
# CELL 8 - TrainingArguments + Trainer + train
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
args = TrainingArguments(
    output_dir=config.OUTPUT_DIR,
    learning_rate=config.LEARNING_RATE,
    num_train_epochs=config.NUM_EPOCHS,
    per_device_train_batch_size=config.TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=config.EVAL_BATCH_SIZE,
    warmup_ratio=config.WARMUP_RATIO,
    weight_decay=config.WEIGHT_DECAY,
    eval_strategy="epoch",            # v5 name (was evaluation_strategy)
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model=config.METRIC_FOR_BEST,
    greater_is_better=True,
    logging_steps=50,
    bf16=use_bf16,
    fp16=torch.cuda.is_available() and not use_bf16,
    seed=config.SEED,
    push_to_hub=True,
    hub_model_id=config.HUB_MODEL_ID,
    report_to="none",                 # W&B off by default
    max_steps=config.SMOKE_MAX_STEPS if config.SMOKE else -1,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    processing_class=tokenizer,       # v5 name (was tokenizer=)
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=config.EARLY_STOPPING_PATIENCE)],
)
trainer.train()

## Evaluate on the held-out test split

In [ ]:
# CELL 9 - held-out test predictions
import os, json
from sklearn.metrics import classification_report, confusion_matrix

pred = trainer.predict(tok_test)
y_true = pred.label_ids
y_pred = pred.predictions.argmax(-1)

ft = {
    "accuracy": accuracy_score(y_true, y_pred),
    "f1_macro": f1_score(y_true, y_pred, average="macro"),
    "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
}
print("fine-tuned (test):", ft)

In [ ]:
# CELL 10 - confusion matrix + per-class report
import matplotlib.pyplot as plt
os.makedirs(config.RESULTS_DIR, exist_ok=True)
target_names = [id2label[i] for i in range(config.NUM_LABELS)]

report = classification_report(y_true, y_pred, target_names=target_names, digits=4)
with open(f"{config.RESULTS_DIR}/classification_report.txt", "w") as f:
    f.write(report)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(cm, cmap="Blues")
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Banking77 confusion matrix")
fig.colorbar(im); fig.tight_layout()
fig.savefig(f"{config.RESULTS_DIR}/confusion_matrix.png", dpi=120)

cm_off = cm.copy(); np.fill_diagonal(cm_off, 0)
pairs = np.dstack(np.unravel_index(np.argsort(cm_off.ravel())[::-1][:8], cm.shape))[0]
print("top confusions:", [(id2label[i], id2label[j], int(cm[i, j])) for i, j in pairs])

## Baselines & push

In [ ]:
# CELL 11 - baselines (majority class + frozen-encoder linear probe)
from collections import Counter
from sklearn.linear_model import LogisticRegression
from transformers import AutoModel

# (a) majority-class baseline
majority = Counter(train_ds[config.LABEL_COLUMN]).most_common(1)[0][0]
maj_pred = np.full_like(y_true, majority)
maj = {
    "accuracy": accuracy_score(y_true, maj_pred),
    "f1_macro": f1_score(y_true, maj_pred, average="macro", zero_division=0),
    "f1_weighted": f1_score(y_true, maj_pred, average="weighted", zero_division=0),
}

# (b) frozen-encoder linear probe: mean-pooled base embeddings -> logistic regression
base = AutoModel.from_pretrained(config.MODEL_ID, attn_implementation="sdpa").to(model.device).eval()

@torch.no_grad()
def embed(dataset):
    feats = []
    loader = trainer.get_eval_dataloader(dataset)   # sequential, reuses collator
    for batch in loader:
        batch = {k: v.to(base.device) for k, v in batch.items() if k != "labels"}
        out = base(**batch).last_hidden_state
        mask = batch["attention_mask"].unsqueeze(-1).float()
        pooled = (out * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        feats.append(pooled.float().cpu().numpy())
    return np.concatenate(feats)

Xtr, ytr = embed(tok_train), np.array(train_ds[config.LABEL_COLUMN])
Xte = embed(tok_test)
probe = LogisticRegression(max_iter=2000, n_jobs=-1).fit(Xtr, ytr)
pp = probe.predict(Xte)
frozen = {
    "accuracy": accuracy_score(y_true, pp),
    "f1_macro": f1_score(y_true, pp, average="macro"),
    "f1_weighted": f1_score(y_true, pp, average="weighted"),
}
print("majority:", maj); print("frozen probe:", frozen); print("fine-tuned:", ft)

In [ ]:
# CELL 12 - write metrics.json (source of truth for the card)
metrics = {
    "dataset": config.DATASET_ID,
    "base_model": config.MODEL_ID,
    "test_size": int(len(y_true)),
    "majority_class": {"name": id2label[majority], **maj},
    "frozen_linear_probe": frozen,
    "fine_tuned": ft,
}
with open(f"{config.RESULTS_DIR}/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

In [ ]:
# CELL 13 - push model + tokenizer to the Hub
trainer.push_to_hub(commit_message="Fine-tuned ModernBERT-base on Banking77")